In [4]:
import pandas as pd
import numpy as np

from scipy import stats

import sys
sys.path.append("../src")

from hypothesis_tests import *

In [5]:
df = pd.read_csv(
    "../data/raw/insurance_data.csv"
)

df.head()

,CustomerID,Age,Gender,Province,VehicleType,AnnualIncome,RiskScore,AnnualPremium,Deductible,NCD,...,Claimed,ClaimAmount,TotalPremium,TotalClaims,CoverType,AutoMake,VehicleModel,CustomValueEstimate,ZipCode,TransactionDate
0,AC-100000,56,Male,Addis Ababa,Sedan,147270,61,2346,500,30,...,False,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10
1,AC-100001,69,Female,Addis Ababa,SUV,74640,57,2334,500,0,...,True,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13
2,AC-100002,46,Male,Oromia,Sedan,70555,42,1697,250,20,...,False,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17
3,AC-100003,32,Female,Somali,Sedan,89398,63,2370,500,20,...,True,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17
4,AC-100004,60,Female,Tigray,SUV,78475,69,2582,500,0,...,False,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10


In [6]:
df["ClaimFrequency"] = (
    df["Claimed"]
    .astype(int)
)


df["Margin"] = (
    df["TotalPremium"]
    -
    df["TotalClaims"]
)

Hypothesis 1: Province Risk Difference
H0:

There is no difference in claim frequency across provinces.

Create table:

In [7]:
province_claims = pd.crosstab(
    df["Province"],
    df["Claimed"]
)

province_claims

Claimed,False,True
Province,,
Addis Ababa,3008,559
Amhara,1720,279
Oromia,2069,377
Somali,977,207
Tigray,691,113


In [8]:
chi2, p, dof, expected = stats.chi2_contingency(
    province_claims
)

print("p-value:", p)

p-value: 0.07608907812609692


In [9]:
decision(p)

'Fail to reject H0'

Hypothesis 2: Gender Risk Difference

In [10]:
gender_claims = pd.crosstab(
    df["Gender"],
    df["Claimed"]
)

chi2,p,dof,expected = stats.chi2_contingency(
    gender_claims
)

print(p)

decision(p)

0.9638306173980254


'Fail to reject H0'

Hypothesis 3: Zip Code Risk Difference

Because ZipCode has many categories, compare two largest groups.

In [11]:
df["ZipCode"].value_counts().head()

ZipCode
10004    733
10002    732
10003    714
10001    710
10005    678
Name: count, dtype: int64

In [12]:
zip_a = df[
    df["ZipCode"] == 10002
]["ClaimFrequency"]

zip_b = df[
    df["ZipCode"] == 40005
]["ClaimFrequency"]

In [13]:
stats.ttest_ind(
    zip_a,
    zip_b
)

TtestResult(statistic=np.float64(-0.919706765042076), pvalue=np.float64(0.35796630862132944), df=np.float64(923.0))

Hypothesis 4: Zip Code Margin Difference

In [14]:
margin_a = df[
    df["ZipCode"] == 10002
]["Margin"]


margin_b = df[
    df["ZipCode"] == 40005
]["Margin"]


stats.ttest_ind(
    margin_a,
    margin_b,
    equal_var=False
)

TtestResult(statistic=np.float64(0.7065553616279882), pvalue=np.float64(0.48036785903449913), df=np.float64(313.3611228659556))

In [15]:
results = pd.DataFrame({
    "Hypothesis":[
        "Province risk",
        "Gender risk",
        "Zip risk",
        "Zip margin"
    ],

    "Test":[
        "Chi-square",
        "Chi-square",
        "T-test",
        "T-test"
    ],

    "Decision":[
        "",
        "",
        "",
        ""
    ]
})

results

,Hypothesis,Test,Decision
0,Province risk,Chi-square,
1,Gender risk,Chi-square,
2,Zip risk,T-test,
3,Zip margin,T-test,
